# MP1a CourtListener — interactive figures (notebook)

This notebook renders the **same Plotly figures** as `courtlistener_mp1a_dash.py` so you can **rotate 3D scenes, zoom, and hover** inside Jupyter / VS Code without starting the Dash server.

**Dash + Mantine Components** (sliders, cards, alerts, badges) still live in the full app:

```bash
cd "week 6" && .venv/bin/python courtlistener_mp1a_dash.py
```

Then open `http://127.0.0.1:8050`.

**Kernel:** use the Week 6 venv (`.venv`) so `pandas`, `plotly`, and `ipywidgets` are available. The notebook imports only the Plotly helpers from `courtlistener_mp1a_dash.py`; `dash_mantine_components` is loaded only when you run the Dash server.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path


def find_mp1a_dir() -> Path:
    """Find folder containing courtlistener_mp1a_dash.py + CSV (cwd may be week 6 or repo root)."""
    here = Path.cwd().resolve()
    candidates = [here, here / "week 6"]
    for p in here.parents:
        candidates.append(p / "week 6")
    for c in candidates:
        dash_py = c / "courtlistener_mp1a_dash.py"
        csv_path = c / "Week6_files" / "courtlistener_week5_repull_40k.csv"
        if dash_py.is_file() and csv_path.is_file():
            return c
    raise FileNotFoundError(
        "Could not find courtlistener_mp1a_dash.py and Week6_files CSV. "
        "Set the notebook working directory to `week 6` or the repo root."
    )


print("Python:", sys.executable)

ROOT = find_mp1a_dir()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import courtlistener_mp1a_dash as mp1a

print("MP1a root:", ROOT)

In [5]:
df = mp1a.load_df()
print(f"Loaded {len(df):,} rows")
df[["court_id", "cite_count"]].groupby("court_id").agg(["mean", "sum", "count"]).round(4)

NameError: name 'mp1a' is not defined

## Question (a) — mean cites by month × court (3D)

Use the **mode bar** (top-right of the figure) to zoom; **click-drag** the 3D scene to rotate.

In [6]:
fig_a = mp1a.fig_q1_scatter3d_monthly(df)
fig_a.show(config=mp1a._GRAPH_CONFIG)

NameError: name 'mp1a' is not defined

## Question (b) — jurisdiction slice, 3D rank × cite × row depth

Controls below mirror the Dash **Select / SegmentedControl / Slider** (Mantine in the app). Changing a control re-builds the figure.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

court_dd = widgets.Dropdown(
    options=[("Federal — W.D. Wash. (wawd)", "wawd"), ("State — Wash. Ct. App. (washctapp)", "washctapp")],
    value="wawd",
    description="Court",
    style={"description_width": "initial"},
    layout=widgets.Layout(min_width="280px"),
)
metric_rb = widgets.RadioButtons(
    options=[("max cite per title", "max"), ("mean cite per title", "mean")],
    description="Metric",
    style={"description_width": "initial"},
)
topn_b = widgets.IntSlider(value=45, min=5, max=60, step=1, description="Top N titles")
out_b = widgets.Output()


def redraw_b(_=None):
    with out_b:
        clear_output(wait=True)
        fig = mp1a.fig_q2_scatter3d_jurisdiction(
            df,
            court_id=court_dd.value,
            top_n=topn_b.value,
            rank_mode=metric_rb.value,
        )
        fig.show(config=mp1a._GRAPH_CONFIG)


for w in (court_dd, metric_rb, topn_b):
    w.observe(redraw_b, names="value")

display(widgets.VBox([widgets.HBox([court_dd, topn_b]), metric_rb, out_b]))
redraw_b()

## Question (c) — top authorities by summed cites (horizontal bars)

Interactive **pan/zoom** on the bar chart; hover for exact values. Sliders match the Mantine sliders in the Dash app.

In [ ]:
topn_c = widgets.IntSlider(value=45, min=5, max=60, step=1, description="Top N titles")
min_rows_c = widgets.IntSlider(value=1, min=1, max=50, step=1, description="Min rows / title")
out_c = widgets.Output()


def redraw_c(_=None):
    with out_c:
        clear_output(wait=True)
        fig = mp1a.fig_q3_top_authorities(
            df, top_n=topn_c.value, min_opinion_rows=min_rows_c.value
        )
        fig.show(config=mp1a._GRAPH_CONFIG)


topn_c.observe(redraw_c, names="value")
min_rows_c.observe(redraw_c, names="value")

display(widgets.VBox([widgets.HBox([topn_c, min_rows_c]), out_c]))
redraw_c()